# 03 — Modeling

**What this notebook is for.** We take the flat feature table from
`02_feat_agg.ipynb` and build models to predict 30-day readmission, following
the agreed plan:

1. **Logistic regression** — a simple baseline to compare everything against.
2. **Raw CatBoost** — a strong tree model with default settings, no imbalance help.
3. **CatBoost + imbalance handling** — check whether class weights (and optionally
   downsampling) actually improve things.
4. **Tune** the chosen setup to get the final model.

**How we measure.** Lead metric is **AUROC** (the project's metric), with
**PR-AUC** reported alongside — PR-AUC is more sensitive to the rare positive
class, so it's where imbalance methods (step 3) show their real effect, even when
AUROC barely moves.

**How we validate — patient-grouped cross-validation.** The EDA found the same
patient can appear in several admissions. If one patient's admissions land in
both the training and validation parts of a fold, the model gets an unfairly easy
test. So every fold keeps each patient wholly on one side (`StratifiedGroupKFold`
grouped on `subject_id`), while also keeping the ~18% positive rate even across
folds.

**How to read this notebook.** Every code cell is followed by a *"How to read
this"* note in plain language.

## 1. Load the feature table

We load the flat table built in notebook 02. It has bookkeeping columns
(`id`, `subject_id`, `split`, `y`) plus the feature columns. Recall the table
contains train + valid only (we don't touch the project's test set here).

In [1]:
import os, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)

FEAT_PATH = os.path.join("data", "features_track1.parquet")
data = pd.read_parquet(FEAT_PATH)

print("feature table shape:", data.shape)
print(data["split"].value_counts())

BOOKKEEPING = ["id", "subject_id", "split", "y"]
feature_names = [c for c in data.columns if c not in BOOKKEEPING]
print("number of feature columns:", len(feature_names))

feature table shape: (11022, 251)
split
train    8234
valid    2788
Name: count, dtype: int64
number of feature columns: 247


**How to read this:** we loaded the table and separated the four bookkeeping
columns from the real features. **Remember:** `y` is the label (0/1),
`subject_id` is what we group folds by, and `split` marks the original
train/valid origin. Only `feature_names` go into the models.

## 2. Set up X, y, and the patient groups

**Decision — we pool train + valid for cross-validation.** The dataset is small
(~11k admissions), and the project's real hold-out is the ungraded `test`
submission, so we use all labeled data for CV rather than carving out yet another
hold-out. (If you'd rather keep `valid` as a one-time final check, you would
instead fit with CV on the `train` rows only and score `valid` once at the end —
a valid alternative, just with less data for CV.)

In [2]:
X = data[feature_names].copy()
y = data["y"].astype(int).values
groups = data["subject_id"].values      # for patient-grouped folds

print("X shape:", X.shape)
print("positive rate:", round(y.mean(), 4))
print("unique patients:", len(np.unique(groups)))

# which features are categorical (matters for both models)
CATEGORICAL = ["gender", "ethnicity"]
CATEGORICAL = [c for c in CATEGORICAL if c in feature_names]  # keep only those that survived
NUMERIC_NON_FLAG = [c for c in ["age", "sequence_length"] if c in feature_names]
print("categorical features:", CATEGORICAL)

X shape: (11022, 247)
positive rate: 0.1751
unique patients: 8833
categorical features: ['gender', 'ethnicity']


**How to read this:** `X` is the feature matrix, `y` the labels, `groups` the
patient ids used to keep folds patient-clean. We also note which columns are
categorical (`gender`, `ethnicity`) — the two models treat these differently, so
we'll handle each in its own section. **Remember:** the positive rate (~0.18)
confirms the imbalance we're designing around.

## 3. One shared cross-validation splitter

We build the fold splitter **once** and reuse it for every model, so all models
are scored on the exact same folds. Comparing models on identical splits means
any difference is the model, not luck of the split. We also write one small
helper that runs a model through the folds and reports mean ± std AUROC and
PR-AUC.

In [3]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score

N_SPLITS = 5
cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# precompute the fold indices once so every model uses identical splits
folds = list(cv.split(X, y, groups))

# sanity: confirm no patient appears in both train and val of any fold
for k, (tr, va) in enumerate(folds):
    overlap = set(groups[tr]) & set(groups[va])
    assert len(overlap) == 0, f"patient leak in fold {k}"
print(f"{N_SPLITS} folds ready, no patient overlap in any fold.")

def evaluate(make_model, X, y, folds, preprocess=None, cat_features=None):
    """Train/score a model across folds. make_model() returns a fresh model.
    preprocess: optional function(X_train, X_val) -> (X_train_p, X_val_p) fit on train only.
    Returns arrays of per-fold AUROC and PR-AUC."""
    aucs, prs = [], []
    for tr, va in folds:
        X_tr, X_va = X.iloc[tr], X.iloc[va]
        y_tr, y_va = y[tr], y[va]
        if preprocess is not None:
            X_tr, X_va = preprocess(X_tr, X_va)
        model = make_model()
        if cat_features is not None:
            model.fit(X_tr, y_tr, cat_features=cat_features)
        else:
            model.fit(X_tr, y_tr)
        p = model.predict_proba(X_va)[:, 1]
        aucs.append(roc_auc_score(y_va, p))
        prs.append(average_precision_score(y_va, p))
    return np.array(aucs), np.array(prs)

def report(name, aucs, prs):
    print(f"{name:32s} AUROC {aucs.mean():.4f} ± {aucs.std():.4f}"
          f" | PR-AUC {prs.mean():.4f} ± {prs.std():.4f}")
    return {"model": name, "auroc": aucs.mean(), "auroc_std": aucs.std(),
            "prauc": prs.mean(), "prauc_std": prs.std()}

5 folds ready, no patient overlap in any fold.


**How to read this:** we made 5 patient-grouped, label-stratified folds and
checked that no patient leaks across any fold. The `evaluate` helper runs a model
through those folds; `report` prints mean ± std for both metrics. **Remember:**
`preprocess` is fit **inside each fold on the training part only** — this is how
we avoid leaking information from the validation rows into scaling or encoding.
We'll collect every model's result in a list to compare at the end.

In [4]:
results = []   # we'll append each model's summary here

**How to read this:** an empty list to gather results as we go.

## 4. Step 1 — Logistic Regression baseline

A simple linear model as the floor. Logistic regression needs more care than the
trees:

- **Scale** the numeric features (age, sequence_length, med_intensity) so their
  large ranges don't dominate the 0/1 flags.
- **One-hot** encode `ethnicity` (a 7-way code) so the model doesn't read the
  codes as quantities.
- **Balance** the classes with `class_weight="balanced"` for the ~18% positive rate.
- **log-transform** `sequence_length` to tame its heavy tail (a few 200+ day stays).

All of this is fit inside each fold on the training part only.

In [5]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# log-transform sequence_length up front (safe: same transform for all rows,
# learns nothing from the data)
X_lr = X.copy()
if "sequence_length" in X_lr.columns:
    X_lr["sequence_length"] = np.log1p(X_lr["sequence_length"])

onehot_cols = [c for c in ["ethnicity"] if c in X_lr.columns]
scale_cols  = [c for c in ["age", "sequence_length"] + [c for c in X_lr.columns if c.startswith("med_intensity__")]
               if c in X_lr.columns]
# everything else (the 0/1 flags, gender, the frac columns) passes through as-is

pre = ColumnTransformer(
    transformers=[
        ("scale", StandardScaler(), scale_cols),
        ("onehot", OneHotEncoder(handle_unknown="ignore"), onehot_cols),
    ],
    remainder="passthrough",
)

def make_logreg():
    return Pipeline([
        ("pre", pre),
        ("clf", LogisticRegression(max_iter=2000, class_weight="balanced")),
    ])

# the pipeline handles preprocessing internally, so no external preprocess fn needed
aucs, prs = evaluate(make_logreg, X_lr, y, folds)
results.append(report("Logistic Regression (baseline)", aucs, prs))

Logistic Regression (baseline)   AUROC 0.7770 ± 0.0036 | PR-AUC 0.5734 ± 0.0182


**How to read this:** this is your baseline number — what a simple linear model
achieves on these features. **Remember:** the pipeline refits the scaler and
one-hot encoder inside every fold automatically, so there's no leakage. Whatever
AUROC this gets is the floor CatBoost needs to clearly beat to justify its extra
complexity.

## 5. Step 2 — Raw CatBoost (no imbalance handling)

Now a strong tree model with **default settings and no class weighting** — a
clean reference for "what does CatBoost do untouched." Trees need no scaling or
one-hot: CatBoost handles the categorical columns natively, we just tell it which
ones. We keep iterations moderate and let it run.

In [6]:
from catboost import CatBoostClassifier

def make_catboost_raw():
    return CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        random_seed=42,
        verbose=False,
        # NO class weighting here — this is the untouched reference
    )

aucs, prs = evaluate(make_catboost_raw, X, y, folds, cat_features=CATEGORICAL)
results.append(report("CatBoost (raw)", aucs, prs))

CatBoost (raw)                   AUROC 0.8025 ± 0.0068 | PR-AUC 0.6308 ± 0.0227


**How to read this:** compare this to both the baseline and the ~0.78 figure
from the EDA. If raw CatBoost already scores well on AUROC, that tells us the
imbalance may matter less than expected for our ranking metric — which sets up
the next step. **Remember:** we passed `cat_features` so CatBoost treats
`gender`/`ethnicity` correctly without any encoding on our part.

## 6. Step 3 — CatBoost with imbalance handling

We test whether helping with the class imbalance improves things. Two important
points established in planning:

- **AUROC may barely move.** AUROC measures *ranking* (do positives score higher
  than negatives), which imbalance methods often don't change much. **PR-AUC** is
  where the effect shows, so watch that column.
- **Keep all the data.** We prefer **class weights** over downsampling, because
  downsampling throws away real negative rows we can't spare (~11k total). We show
  downsampling too, but done **inside each fold** so no leakage — and mostly to
  confirm it doesn't beat weights.

### 6a. CatBoost with class weights

In [7]:
def make_catboost_weighted():
    return CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        random_seed=42,
        verbose=False,
        auto_class_weights="Balanced",   # <-- the imbalance handling
    )

aucs, prs = evaluate(make_catboost_weighted, X, y, folds, cat_features=CATEGORICAL)
results.append(report("CatBoost (class weights)", aucs, prs))

CatBoost (class weights)         AUROC 0.7964 ± 0.0047 | PR-AUC 0.6244 ± 0.0247


**How to read this:** compare AUROC and PR-AUC against raw CatBoost. If AUROC is
flat but PR-AUC rises, the weighting helped the rare-class ranking without moving
the headline metric — a normal and useful finding.

### 6b. CatBoost with downsampling (inside each fold)

Downsampling randomly drops majority-class (non-readmitted) rows from the
**training** part of each fold to balance the classes. We try a couple of ratios.
Crucially this happens *after* the fold split, on training rows only — never on
the validation rows.

In [8]:
def evaluate_downsample(make_model, X, y, folds, ratio, cat_features=None, seed=42):
    rng = np.random.RandomState(seed)
    aucs, prs = [], []
    for tr, va in folds:
        X_tr, X_va = X.iloc[tr], X.iloc[va]
        y_tr, y_va = y[tr], y[va]
        # indices of each class within the training fold
        pos_idx = np.where(y_tr == 1)[0]
        neg_idx = np.where(y_tr == 0)[0]
        keep_neg = rng.choice(neg_idx, size=min(len(neg_idx), int(len(pos_idx) * ratio)), replace=False)
        keep = np.concatenate([pos_idx, keep_neg])
        rng.shuffle(keep)
        X_tr_ds, y_tr_ds = X_tr.iloc[keep], y_tr[keep]
        model = make_model()
        if cat_features is not None:
            model.fit(X_tr_ds, y_tr_ds, cat_features=cat_features)
        else:
            model.fit(X_tr_ds, y_tr_ds)
        p = model.predict_proba(X_va)[:, 1]
        aucs.append(roc_auc_score(y_va, p))
        prs.append(average_precision_score(y_va, p))
    return np.array(aucs), np.array(prs)

for ratio in [1.0, 2.0, 3.0]:   # 1:1, 1:2, 1:3 positive:negative
    aucs, prs = evaluate_downsample(make_catboost_raw, X, y, folds, ratio, cat_features=CATEGORICAL)
    results.append(report(f"CatBoost (downsample 1:{int(ratio)})", aucs, prs))

CatBoost (downsample 1:1)        AUROC 0.7963 ± 0.0070 | PR-AUC 0.6109 ± 0.0254
CatBoost (downsample 1:2)        AUROC 0.8003 ± 0.0084 | PR-AUC 0.6216 ± 0.0233
CatBoost (downsample 1:3)        AUROC 0.8005 ± 0.0060 | PR-AUC 0.6279 ± 0.0228


**How to read this:** each row downsamples negatives to a different ratio,
inside the fold. Compare to class weights and raw. **Remember — the likely
outcome:** AUROC stays about the same or dips slightly (downsampling discards
data), while PR-AUC moves around. If nothing beats class weights on the metrics
you care about, the right call is to **keep all data with class weights** (or even
stay raw) and move on — a legitimate result, not a failure.

## 7. Step 4 — Tune the chosen configuration

Look at the table below (built in the next section) and pick the configuration to
carry forward — likely raw CatBoost or CatBoost with class weights, since
downsampling rarely wins on AUROC. Then tune its most important parameters with a
small grid on the **same folds**. We keep the search small on purpose: with ~11k
rows and 5-fold grouped CV, a huge sweep is slow and can overfit to the folds.

The parameters that matter most for CatBoost:
- `depth` — tree depth / model complexity (try 4, 6, 8).
- `learning_rate` — smaller learns slower but often generalizes better.
- `l2_leaf_reg` — regularization strength (higher fights overfitting).

Edit `BEST_IMBALANCE` below to match your choice from the table, then run.

In [9]:
from itertools import product

# ---- set this to the winner from the results table ----
# options: "raw" (no weighting) or "balanced" (auto_class_weights)
BEST_IMBALANCE = "raw"

def make_tuned(depth, lr, l2):
    kwargs = dict(iterations=1500, learning_rate=lr, depth=depth, l2_leaf_reg=l2,
                  random_seed=42, verbose=False, early_stopping_rounds=50)
    if BEST_IMBALANCE == "balanced":
        kwargs["auto_class_weights"] = "Balanced"
    return CatBoostClassifier(**kwargs)

# wider grid: spans genuinely shallow->deep and weak->strong regularization,
# with low learning rates paired to many iterations + early stopping
grid = {
    "depth": [3, 5, 7],
    "learning_rate": [0.02, 0.05, 0.1],
    "l2_leaf_reg": [1, 5, 20],
}

tuning_rows = []
best = None
for depth, lr, l2 in product(grid["depth"], grid["learning_rate"], grid["l2_leaf_reg"]):
    aucs, prs = evaluate(lambda: make_tuned(depth, lr, l2), X, y, folds, cat_features=CATEGORICAL)
    row = {"depth": depth, "learning_rate": lr, "l2_leaf_reg": l2,
           "auroc": aucs.mean(), "auroc_std": aucs.std(), "prauc": prs.mean()}
    tuning_rows.append(row)
    if best is None or row["auroc"] > best["auroc"]:
        best = row
    print(f"depth={depth} lr={lr} l2={l2}  ->  AUROC {aucs.mean():.4f} ± {aucs.std():.4f}  PR-AUC {prs.mean():.4f}")

print("\nbest by AUROC:", best)

depth=3 lr=0.02 l2=1  ->  AUROC 0.7972 ± 0.0052  PR-AUC 0.6195
depth=3 lr=0.02 l2=5  ->  AUROC 0.7967 ± 0.0052  PR-AUC 0.6190
depth=3 lr=0.02 l2=20  ->  AUROC 0.7974 ± 0.0068  PR-AUC 0.6194
depth=3 lr=0.05 l2=1  ->  AUROC 0.7964 ± 0.0058  PR-AUC 0.6143
depth=3 lr=0.05 l2=5  ->  AUROC 0.7978 ± 0.0055  PR-AUC 0.6180
depth=3 lr=0.05 l2=20  ->  AUROC 0.7979 ± 0.0071  PR-AUC 0.6197
depth=3 lr=0.1 l2=1  ->  AUROC 0.7894 ± 0.0049  PR-AUC 0.6047
depth=3 lr=0.1 l2=5  ->  AUROC 0.7916 ± 0.0072  PR-AUC 0.6088
depth=3 lr=0.1 l2=20  ->  AUROC 0.7964 ± 0.0084  PR-AUC 0.6167
depth=5 lr=0.02 l2=1  ->  AUROC 0.8022 ± 0.0059  PR-AUC 0.6295
depth=5 lr=0.02 l2=5  ->  AUROC 0.8037 ± 0.0078  PR-AUC 0.6301
depth=5 lr=0.02 l2=20  ->  AUROC 0.8030 ± 0.0067  PR-AUC 0.6300
depth=5 lr=0.05 l2=1  ->  AUROC 0.7927 ± 0.0084  PR-AUC 0.6194
depth=5 lr=0.05 l2=5  ->  AUROC 0.7970 ± 0.0083  PR-AUC 0.6211
depth=5 lr=0.05 l2=20  ->  AUROC 0.8015 ± 0.0079  PR-AUC 0.6288
depth=5 lr=0.1 l2=1  ->  AUROC 0.7866 ± 0.0083  PR-AU

In [10]:
# re-run the best config once to get its TRUE per-fold arrays (the earlier
# `best` dict only held the averaged mean, which is why std showed as 0.0000)
best_aucs, best_prs = evaluate(
    lambda: make_tuned(best["depth"], best["learning_rate"], best["l2_leaf_reg"]),
    X, y, folds, cat_features=CATEGORICAL
)
results.append(report(
    f"CatBoost (tuned: d{best['depth']} lr{best['learning_rate']} l2{best['l2_leaf_reg']})",
    best_aucs, best_prs
))

CatBoost (tuned: d7 lr0.02 l220) AUROC 0.8042 ± 0.0075 | PR-AUC 0.6347 ± 0.0211


**How to read this:** each line is one parameter combination scored on the same
folds; we keep the best by AUROC. **Remember:** if the tuned score is only a
hair above untuned CatBoost, that's telling you the defaults were already good and
the features (not tuning) carry the signal — a common and honest outcome. Don't
over-invest in squeezing tiny gains.

## 8. Compare all models

The whole point of running these on identical folds: one clean comparison table.
AUROC is the lead metric; PR-AUC shows the imbalance story.

In [11]:
summary = pd.DataFrame(results)[["model", "auroc", "auroc_std", "prauc", "prauc_std"]]
summary = summary.sort_values("auroc", ascending=False).reset_index(drop=True)
print(summary.round(4).to_string(index=False))

                           model  auroc  auroc_std  prauc  prauc_std
CatBoost (tuned: d7 lr0.02 l220) 0.8042     0.0075 0.6347     0.0211
                  CatBoost (raw) 0.8025     0.0068 0.6308     0.0227
       CatBoost (downsample 1:3) 0.8005     0.0060 0.6279     0.0228
       CatBoost (downsample 1:2) 0.8003     0.0084 0.6216     0.0233
        CatBoost (class weights) 0.7964     0.0047 0.6244     0.0247
       CatBoost (downsample 1:1) 0.7963     0.0070 0.6109     0.0254
  Logistic Regression (baseline) 0.7770     0.0036 0.5734     0.0182


**How to read this:** top row is your best model by AUROC. Read the ladder:
how far above the logistic baseline is CatBoost (what the model bought you), and
did any imbalance method actually beat raw/weighted on AUROC (usually not).
**Remember:** compare the winner to the ~0.78 EDA sanity figure — you should be
at or above it. The `std` columns tell you how stable each model is across folds;
a big std means the score depends a lot on which patients landed where.

In [13]:
## --- Multi-seed stability check on the CHOSEN model only ---
# Re-runs ONE model across several CV seeds. Each seed reshuffles patients into
# different folds, so this measures how much the headline AUROC moves purely due
# to how the split fell — the variance the single-split "± std" can't see.

from sklearn.model_selection import StratifiedGroupKFold

# ================== SET YOUR WINNING MODEL HERE ==================
# Return a FRESH model each call. Edit this to match the config you chose.
# Example shown: raw CatBoost (what you found best). To evaluate the tuned
# config instead, swap in depth/learning_rate/l2_leaf_reg and (if you chose it)
# auto_class_weights="Balanced".
def make_chosen_model():
    return CatBoostClassifier(
        iterations=500,
        learning_rate=0.02,
        depth=7,
        random_seed=42,
        verbose=False,
        l2_leaf_reg=20
        # auto_class_weights="Balanced",   # uncomment if your winner used weights
    )
CHOSEN_NAME = "CatBoost (raw)"
# ================================================================

SEEDS = [0, 1, 2, 7, 42, 123, 2024]   # 7 different fold partitions

seed_aucs, seed_prs = [], []
for s in SEEDS:
    cv_s = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=s)
    folds_s = list(cv_s.split(X, y, groups))
    aucs, prs = evaluate(make_chosen_model, X, y, folds_s, cat_features=CATEGORICAL)
    seed_aucs.append(aucs.mean())   # this seed's 5-fold mean
    seed_prs.append(prs.mean())
    print(f"seed {s:>4}:  AUROC {aucs.mean():.4f}   PR-AUC {prs.mean():.4f}")

seed_aucs, seed_prs = np.array(seed_aucs), np.array(seed_prs)
print(f"\n{CHOSEN_NAME} across {len(SEEDS)} seeds:")
print(f"  AUROC : mean {seed_aucs.mean():.4f}  std {seed_aucs.std():.4f}"
      f"  range [{seed_aucs.min():.4f}, {seed_aucs.max():.4f}]")
print(f"  PR-AUC: mean {seed_prs.mean():.4f}  std {seed_prs.std():.4f}"
      f"  range [{seed_prs.min():.4f}, {seed_prs.max():.4f}]")

seed    0:  AUROC 0.8034   PR-AUC 0.6276
seed    1:  AUROC 0.8051   PR-AUC 0.6288
seed    2:  AUROC 0.8029   PR-AUC 0.6258
seed    7:  AUROC 0.8014   PR-AUC 0.6235
seed   42:  AUROC 0.8024   PR-AUC 0.6292
seed  123:  AUROC 0.8040   PR-AUC 0.6267
seed 2024:  AUROC 0.8035   PR-AUC 0.6265

CatBoost (raw) across 7 seeds:
  AUROC : mean 0.8032  std 0.0011  range [0.8014, 0.8051]
  PR-AUC: mean 0.6269  std 0.0018  range [0.6235, 0.6292]


## 9. What's next

- **Feature importance / SHAP** on the chosen model, to sanity-check that the top
  drivers match the EDA (age, med_count, a few labs) and to consider pruning weak
  features.
- **Track 2 (sequence model)** if you want to test whether temporal structure in
  the labs beats this flat table — it must beat the number above to be worth it.
- **Final submission** (separate, careful step): retrain the chosen model on all
  labeled data and produce predictions for the project's `test.csv`. Its AUROC
  isn't visible to us, so we rely on the CV estimate here as our best guess of
  how it will do.

*Reminder:* nothing here touched the project's `test` set — these are all
cross-validated estimates on the labeled train + valid data.